# Lab 3 — Multi-expiry options market making, Heston, 100 options
Slide 24: with $(S,\sigma^2)\sim$ Heston the value function is $v(t,S_t,Q_t,\sigma_t^2)$ (Baldacci et al., 2021). Book of five expiries $\times$ twenty strikes, marks at constant implied vols, delta-hedged book value as in Lab 2 with $\sigma_t^2 = v_t$:
$$dS_t/S_t = \sqrt{v_t}\,dW^1_t,\qquad dv_t = \kappa_v(\theta_v - v_t)dt + \xi\sqrt{v_t}\,dW^2_t,\qquad d\langle W^1,W^2\rangle_t=\rho\,dt.$$

In [ ]:
import math, torch, numpy as np, matplotlib.pyplot as plt
from omm import *
torch.manual_seed(0)

S0, tau, N = 100.0, 1.0, 390; dt = tau / N
v0, th, kv, xi, rho = 0.045 / 252, 0.045 / 252, 2.0 / 252, 0.5 / 252, -0.7   # Heston in daily units (annual: v0 = theta = 0.045, kappa_v = 2, xi = 0.5)
Ts = torch.tensor([10.0, 21.0, 42.0, 63.0, 126.0], device=DEV)              # expiries (trading days)
z = torch.linspace(-2.0, 2.0, 20, device=DEV)                                # standardised log-moneyness
T, Z = Ts.repeat_interleave(20), z.repeat(5); n = len(T)                     # 100 options
K = S0 * torch.exp(Z * 0.20 * torch.sqrt(T / 252))
sig_imp = (0.20 - 0.02 * Z + 0.01 * Z ** 2) / math.sqrt(252)                # implied surface sigma(K,T), constant over the session
sig_ref = 0.20 / math.sqrt(252)
lam_i = torch.full((n,), 5.0, device=DEV)
_, _, _, vega = bs_call(torch.tensor(S0, device=DEV), K, T, sig_imp)
kappa_i = 1.5 / (0.01 * vega / math.sqrt(252))                               # 1/kappa_i = 0.67 vol points in price
alpha, beta = 1e-3, 1e-3
feat_i = torch.stack([Z / 2, T / 126, sig_imp * math.sqrt(252)], -1)       # per-option features (n,3)
print(DEV, "FAST" if FAST else "full scale")

## Marks and simulation

In [ ]:
def marks(t, S):                                                        # BS price, delta, dollar gamma at the implied vols, (B,n)
    C, D, G, _ = bs_call(S[:, None], K, T - (t * torch.ones_like(S))[:, None], sig_imp)
    return C, D, G * S[:, None] ** 2

def simulate(policy, B, seed=None):
    g = torch.Generator(device=DEV)
    if seed is not None: g.manual_seed(seed)
    Zn = math.sqrt(dt) * torch.randn(B, N, 2, device=DEV, generator=g)
    S, v, Q = torch.full((B,), S0, device=DEV), torch.full((B,), v0, device=DEV), torch.zeros(B, n, device=DEV)
    out = dict(rbar=[], logp=[], cost=[], dN=[], Q=[Q]); spread = mkt = run = 0.0
    for k in range(N):
        t = k * dt; _, _, G = marks(t, S)
        d = policy(t, S, v, Q)                                              # (B,n,2)
        lam_dt = lam_i[:, None] * torch.exp(-kappa_i[:, None] * d) * dt
        dN, lp = fills(lam_dt)
        drift = (0.5 * (v[:, None] - sig_imp ** 2) * Q * G).sum(1) * dt      # hedged inventory P&L with sigma_t^2 = v_t
        cost = alpha * (Q ** 2).sum(1) * dt
        out["rbar"].append((d * lam_dt).sum((1, 2))); out["logp"].append(lp.sum((1, 2)))
        out["cost"].append(cost - drift); out["dN"].append(dN)
        spread = spread + (d.detach() * dN).sum((1, 2)); mkt = mkt + drift; run = run + cost
        vp = v.clamp(min=0.0); dW1, dW2 = Zn[:, k, 0], rho * Zn[:, k, 0] + math.sqrt(1 - rho ** 2) * Zn[:, k, 1]
        Q = Q + dN[..., 0] - dN[..., 1]
        S, v = S * torch.exp(torch.sqrt(vp) * dW1 - 0.5 * vp * dt), v + kv * (th - vp) * dt + xi * torch.sqrt(vp) * dW2
        out["Q"].append(Q)
    out = {k: torch.stack(v_, 1) for k, v_ in out.items()}
    out.update(term=beta * (Q ** 2).sum(1), S=S, v=v, spread=spread, mkt=mkt, pnl=spread + mkt)
    out["J"] = out["pnl"] - run - out["term"]
    return out

myopic = lambda t, S, v, Q: (1 / kappa_i)[None, :, None].expand(len(S), n, 2)

## DSO: one quoting network shared across the book
$\Phi$ is applied option by option to $(t, S_t, \sigma_t^2, Q^i_t;\,K_i,T_i,\sigma_i)$ — the restriction of $\Phi(t,S_t,Q_t;\theta)$ to the separable structure of the objective (see the DGM section) — so the same network quotes 100 or 10,000 options.

In [ ]:
def feats(t, S, v, Q):                                                   # (B,n,7)
    B = len(S)
    mkt = torch.stack([t / tau * torch.ones_like(S), torch.log(S / S0) / sig_ref, (v - v0) / v0], -1)[:, None, :].expand(B, n, 3)
    return torch.cat([mkt, Q[..., None] / 5.0, feat_i[None].expand(B, n, 3)], -1)

net = MLP(7, 2, zero_last=True).to(DEV)
dso_policy = lambda t, S, v, Q: 1 / kappa_i[:, None] + ckpt(net, feats(t, S, v, Q).reshape(-1, 7)).view(-1, n, 2)

def dso_step():
    o = simulate(dso_policy, B=scale(1024, 8))
    surr, J = dso_surrogate(o["rbar"], o["logp"], o["cost"], o["term"])
    return -surr, J

hist_dso = train(dso_step, net.parameters(), iters=scale(1500), lr=1e-3, clip=10.0, name="DSO")

## DGM: the HJB for $v(t,S,\sigma^2,Q)$ separates across options
With $|Q|^2=\sum_i (Q^i)^2$, independent fills and a sup that is a sum of per-option sups, $v(t,S,\sigma^2,Q)=\sum_i v_i(t,S,\sigma^2,Q^i)$ where each $v_i$ solves the four-dimensional HJB
$$\partial_t v_i + \mathcal A v_i + \tfrac12(\sigma^2-\sigma_i^2)\,q\,\Gamma^{\$,i}(t,S) - \alpha q^2 + \tfrac{\lambda_i}{\kappa_i}\big(e^{\kappa_i\Delta^b_q v_i-1}+e^{\kappa_i\Delta^a_q v_i-1}\big)=0,\qquad v_i(\tau,\cdot,q)=-\beta q^2,$$
$\mathcal A = \tfrac12\sigma^2S^2\partial_{SS} + \kappa_v(\theta_v-\sigma^2)\partial_{\sigma^2} + \tfrac12\xi^2\sigma^2\partial_{\sigma^2\sigma^2} + \rho\xi\sigma^2 S\,\partial_{S\sigma^2}$ the Heston generator. One network $V^\theta(t,S,\sigma^2,q;\,K_i,T_i,\sigma_i)$ conditioned on the option represents every $v_i$, so DGM handles the whole book at the cost of a single low-dimensional PDE. A penalty on aggregated Greeks instead of $|Q|^2$ would couple the options (Baldacci et al., 2021).

In [ ]:
Qmax = 15
vnet = MLP(7, 1, width=128, depth=3).to(DEV)

def Vf(t, S, v, q, i):                                                    # value of option i's sub-problem, i: (B,) long
    x = torch.stack([t / tau, torch.log(S / S0) / sig_ref, (v - v0) / v0, q / 5.0], -1)
    return vnet(torch.cat([x, feat_i[i]], -1)).squeeze(-1)

def hjb(t, S, v, q, i):
    t, S, v = (x.requires_grad_(True) for x in (t, S, v))
    val = Vf(t, S, v, q, i)
    v_t, v_S, v_v = grad(val, t), grad(val, S), grad(val, v)
    v_SS, v_vv, v_Sv = grad(v_S, S), grad(v_v, v), grad(v_S, v)
    Db, Da = Vf(t, S, v, q + 1, i) - val, Vf(t, S, v, q - 1, i) - val
    _, _, G, _ = bs_call(S, K[i], T[i] - t, sig_imp[i]); G = G * S ** 2
    A = 0.5 * v * S ** 2 * v_SS + kv * (th - v) * v_v + 0.5 * xi ** 2 * v * v_vv + rho * xi * v * S * v_Sv
    H = lam_i[i] / kappa_i[i] * (torch.exp(kappa_i[i] * Db - 1) + torch.exp(kappa_i[i] * Da - 1))
    return v_t + A + 0.5 * (v - sig_imp[i] ** 2) * q * G - alpha * q ** 2 + H

def dgm_step(B=scale(2048, 4), w=10.0):
    t, S, v = tau * torch.rand(B, device=DEV), S0 * torch.exp(2 * sig_ref * torch.randn(B, device=DEV)), v0 * torch.exp(0.5 * torch.randn(B, device=DEV))
    q, i = torch.randint(-Qmax, Qmax + 1, (B,), device=DEV).float(), torch.randint(0, n, (B,), device=DEV)
    loss = hjb(t, S, v, q, i).pow(2).mean() + w * (Vf(torch.full_like(t, tau), S, v, q, i) + beta * q ** 2).pow(2).mean()
    return loss, loss.item()

hist_dgm = train(dgm_step, vnet.parameters(), iters=scale(6000), lr=1e-3, name="DGM")

def dgm_policy(t, S, v, Q):
    B = len(S); i = torch.arange(n, device=DEV).repeat(B); q = Q.reshape(-1)
    tt, SS, vv = (x.repeat_interleave(n) for x in (t * torch.ones_like(S), S, v))
    val = Vf(tt, SS, vv, q, i)
    return torch.stack([1 / kappa_i[i] + val - Vf(tt, SS, vv, q + 1, i), 1 / kappa_i[i] + val - Vf(tt, SS, vv, q - 1, i)], -1).view(B, n, 2)

## Comparison

In [ ]:
policies = {"myopic 1/k": myopic, "DSO": dso_policy, "DGM": dgm_policy}
with torch.no_grad():
    for j, (name, pol) in enumerate(policies.items()):
        o = simulate(pol, B=scale(4000, 8), seed=1)
        report(name, o["pnl"], o["J"], o["Q"][:, -1].norm(dim=1), o["dN"].sum((1, 2, 3)).mean().item(), header=(j == 0))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
with torch.no_grad():                                                     # quote skew (delta^b - delta^a) kappa_i at Q = 0, t = 0: < 0 bids for inventory (sigma_t^2 > sigma_i^2)
    S1, v1, Q0 = torch.tensor([S0], device=DEV), torch.tensor([v0], device=DEV), torch.zeros(1, n, device=DEV)
    for a, (name, pol) in zip(ax[:2], [("DSO", dso_policy), ("DGM", dgm_policy)]):
        r = pol(0.0, S1, v1, Q0)[0]; skew = ((r[:, 0] - r[:, 1]) * kappa_i).view(5, 20).cpu().numpy()
        im = a.imshow(skew, aspect="auto", cmap="RdBu_r", vmin=-abs(skew).max(), vmax=abs(skew).max(), extent=[-2, 2, 4.5, -0.5])
        a.set_yticks(range(5)); a.set_yticklabels([f"{int(x)}d" for x in Ts.cpu()]); a.set_xlabel("standardised log-moneyness z"); a.set_title(f"{name}: skew (delta^b - delta^a) kappa at Q = 0", fontsize=10); plt.colorbar(im, ax=a)
    atm = 20 + 10                                                         # 21-day expiry, z ~ 0
    qq = torch.arange(-15, 16, device=DEV).float(); Qa = torch.zeros(len(qq), n, device=DEV); Qa[:, atm] = qq
    for name, pol, ls in [("DSO", dso_policy, "o"), ("DGM", dgm_policy, "--")]:
        r = (pol(0.0, torch.full_like(qq, S0), torch.full_like(qq, v0), Qa)[:, atm] * kappa_i[atm]).cpu().numpy()
        ax[2].plot(qq.cpu(), r[:, 0], ls, color=COL["bid"], ms=3); ax[2].plot(qq.cpu(), r[:, 1], ls, color=COL["ask"], ms=3)
    ax[2].set_xlabel("inventory in the 21d ATM option"); ax[2].set_title("kappa * delta, 21d ATM (DSO dots, DGM dashed)", fontsize=10)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist_dso, lw=0.8, color=COL["dso"]); ax[0].set_title("DSO  J_hat(delta^theta)", fontsize=10); ax[0].set_xlabel("iteration")
ax[1].semilogy(hist_dgm, lw=0.8, color=COL["dgm"]); ax[1].set_title("DGM  loss", fontsize=10); ax[1].set_xlabel("iteration"); plt.tight_layout()